# 07 · Knowledge graph export (Neo4j)

`chemkg.graph.export` writes node and relationship CSVs plus `load.cypher` to `data/processed/neo4j/`.

| Nodes | Relationships |
|---|---|
| Person (chemists + external advisors), Country (historical labels), Occupation, Domain, Era, Award, Institution | CITIZEN_OF, HAS_OCCUPATION, IN_DOMAIN, BORN_IN_ERA, ADVISED, RECEIVED, WORKED_AT, EDUCATED_AT, SUCCESSOR_OF |

Load: copy the CSVs into Neo4j's `import/` directory and run `load.cypher` (Neo4j 5).

In [1]:
import json, warnings
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from chemkg.data import load, analysable
from chemkg import countries
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
ds = load()
P, A = ds.people, analysable(ds.people)
NAMES = countries.load_iso3().set_index("iso3").name.to_dict()
print(f"{len(P):,} people, {len(A):,} analysable; enrichment loaded: {ds.enriched}")

11,903 people, 11,860 analysable; enrichment loaded: True


In [2]:
from pathlib import Path
from chemkg.paths import PROCESSED
d = PROCESSED / "neo4j"
pd.Series({f.stem: sum(1 for _ in open(f)) - 1 for f in sorted(d.glob("*.csv"))}, name="rows")

advised            5562
awards             3321
born_in_era       10675
citizen_of        11380
countries           291
educated_at       15619
eras                 14
has_occupation    13058
institutions       4619
occupations          46
persons           15853
received          23854
successor_of        148
worked_at         13292
Name: rows, dtype: int64

In [3]:
print((d / "load.cypher").read_text()[:1500])

// Chemists of Wikidata: load script. Copy data/processed/neo4j/*.csv into Neo4j's import dir.
CREATE CONSTRAINT person_qid IF NOT EXISTS FOR (p:Person) REQUIRE p.qid IS UNIQUE;
CREATE CONSTRAINT country_label IF NOT EXISTS FOR (c:Country) REQUIRE c.label IS UNIQUE;
CREATE CONSTRAINT occupation_name IF NOT EXISTS FOR (o:Occupation) REQUIRE o.name IS UNIQUE;
CREATE CONSTRAINT domain_name IF NOT EXISTS FOR (d:Domain) REQUIRE d.name IS UNIQUE;
CREATE CONSTRAINT era_name IF NOT EXISTS FOR (e:Era) REQUIRE e.name IS UNIQUE;
CREATE CONSTRAINT award_qid IF NOT EXISTS FOR (a:Award) REQUIRE a.qid IS UNIQUE;
CREATE CONSTRAINT inst_qid IF NOT EXISTS FOR (i:Institution) REQUIRE i.qid IS UNIQUE;

LOAD CSV WITH HEADERS FROM 'file:///persons.csv' AS r
MERGE (p:Person {qid: r.qid})
SET p.name = r.name, p.birth_year = toInteger(r.birth_year), p.death_year = toInteger(r.death_year),
    p.in_dataset = r.in_dataset = 'true', p.nobel_chemistry = r.nobel_chemistry = 'true',
    p.sex = r.sex, p.wikipedia_ur

### Example queries
```cypher
// Nobel laureates in Chemistry and their academic grandparents
MATCH (gp:Person)-[:ADVISED]->(:Person)-[:ADVISED]->(n:Person {nobel_chemistry: true})
RETURN n.name, collect(DISTINCT gp.name) LIMIT 20;

// Chemists who were citizens of a historical state and its modern successor
MATCH (p:Person)-[:CITIZEN_OF]->(h:Country)-[:SUCCESSOR_OF]->(m:Country)<-[:CITIZEN_OF]-(p)
RETURN h.label, m.label, count(p) ORDER BY count(p) DESC;

// Institutions that employed the most Nobel laureates
MATCH (p:Person {nobel_chemistry: true})-[:WORKED_AT]->(i:Institution)
RETURN i.name, count(p) AS laureates ORDER BY laureates DESC LIMIT 10;
```